# ДЗ MLOps: обучение, TensorBoard, S3 и DVC

В этом ноутбуке обучается базовая модель ResNet18 на `Train_1`, проводится оценка на `Test_1`, затем модель дообучается на `Train_2` и тестируется на `Test_2`. Метрики обучения и тестирования логируются в TensorBoard, артефакты моделей загружаются в локальный S3 MinIO, а регулярное воспроизведение шагов описано в `dvc.yaml`.


## 1. Импорт библиотек и проверка окружения

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
import yaml

os.environ.setdefault('TORCH_HOME', 'cache/torch')
sys.path.append('src')

from run_training import train_stage
from compare_models import compare
from mlops_utils import read_json, read_yaml, create_s3_client

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


torch: 2.8.0+cu128
cuda available: True
device: cuda


## 2. Параметры эксперимента

Все параметры вынесены в `params.yaml`, чтобы один и тот же конфиг использовался и в ноутбуке, и в DVC-пайплайне.

In [2]:
with open('params.yaml', 'r', encoding='utf-8') as file:
    params = yaml.safe_load(file)

params

{'project': {'data_dir': 'ai-vs-human-generated-dataset-hw',
  'artifacts_dir': 'artifacts',
  'reports_dir': 'reports',
  'logs_dir': 'runs'},
 'model': {'architecture': 'resnet18',
  'pretrained': True,
  'freeze_backbone': False,
  'num_classes': 2},
 's3': {'enabled': True,
  'endpoint_url': 'http://localhost:9000',
  'bucket': 'mlops-homework',
  'access_key': 'minioadmin',
  'secret_key': 'minioadmin',
  'base_model_key': 'models/base_model.pt',
  'finetuned_model_key': 'models/finetuned_model.pt'},
 'train_base': {'train_split': 'Train_1',
  'test_split': 'Test_1',
  'epochs': 5,
  'batch_size': 32,
  'lr': 0.0003,
  'weight_decay': 0.0001,
  'step_size': 3,
  'gamma': 0.5,
  'num_workers': 4,
  'seed': 42},
 'finetune': {'train_split': 'Train_2',
  'test_split': 'Test_2',
  'epochs': 3,
  'batch_size': 32,
  'lr': 0.0001,
  'weight_decay': 0.0001,
  'step_size': 2,
  'gamma': 0.5,
  'num_workers': 4,
  'seed': 43}}

## 3. Проверка структуры данных

In [3]:
data_dir = Path(params['project']['data_dir'])
required_splits = ['Train_1', 'Test_1', 'Train_2', 'Test_2']

for split in required_splits:
    split_dir = data_dir / split
    print(split, 'exists =', split_dir.exists())
    if split_dir.exists():
        csv_files = list(split_dir.glob('*.csv'))
        print('  csv:', [p.name for p in csv_files])
        print('  files count:', len(list(split_dir.iterdir())))

Train_1 exists = True
  csv: ['train.csv']
  files count: 2
Test_1 exists = True
  csv: ['test.csv']
  files count: 2
Train_2 exists = True
  csv: ['train.csv']
  files count: 2
Test_2 exists = True
  csv: ['test.csv']
  files count: 2


## 4. Обучение базовой модели на Train_1 и оценка на Test_1

Перед стартом обучения параметры сохраняются в TensorBoard как текстовые записи. На каждой эпохе логируются loss, accuracy, macro F1, macro precision и macro recall для train/test.

In [4]:
base_metrics = train_stage(stage='base', params_path='params.yaml')
base_metrics['final_test_metrics']

base epoch 5: train_loss=0.0670, train_acc=0.9744, train_f1=0.9744, test_loss=0.1015, test_acc=0.9635, test_f1=0.9635


{'accuracy': 0.96347260445334,
 'f1_macro': 0.9634656868121824,
 'precision_macro': 0.9637762242668079,
 'recall_macro': 0.9634568988051064,
 'loss': 0.10152377421211117}

## 5. Дообучение модели на Train_2 и оценка на Test_2

Если `s3.enabled: true`, базовая модель перед дообучением скачивается из MinIO, затем дообученная версия снова загружается в S3.

In [5]:
finetuned_metrics = train_stage(stage='finetune', params_path='params.yaml')
finetuned_metrics['final_test_metrics']

finetune epoch 3: train_loss=0.0557, train_acc=0.9790, train_f1=0.9790, test_loss=0.0898, test_acc=0.9710, test_f1=0.9710


{'accuracy': 0.971,
 'f1_macro': 0.9709872053575628,
 'precision_macro': 0.9715530561338472,
 'recall_macro': 0.9709192729818246,
 'loss': 0.08975216836854816}

## 6. Сравнение двух версий модели

In [6]:
comparison = compare(
    base_path='reports/base_metrics.json',
    finetuned_path='reports/finetuned_metrics.json',
    output_json='reports/comparison.json',
    output_md='reports/comparison.md',
)

rows = pd.DataFrame(comparison['rows'])
rows

,split,metric,base,finetuned,delta
0,train,loss,0.067012,0.055743,-0.011269
1,train,accuracy,0.974382,0.978984,0.004602
2,train,f1_macro,0.974382,0.978978,0.004596
3,train,precision_macro,0.974382,0.978954,0.004572
4,train,recall_macro,0.974382,0.979005,0.004623
5,test,loss,0.101524,0.089752,-0.011772
6,test,accuracy,0.963473,0.971000,0.007527
7,test,f1_macro,0.963466,0.970987,0.007522
8,test,precision_macro,0.963776,0.971553,0.007777
9,test,recall_macro,0.963457,0.970919,0.007462


In [7]:
print(comparison['conclusion'])

После дообучения качество на тестовой выборке выросло по macro F1. Это означает, что модель смогла использовать данные Train_2 и лучше обобщается на Test_2.


## 7. TensorBoard

Логи находятся в папке `runs`. Для просмотра графиков нужно выполнить команду ниже в терминале.

In [8]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

print('tensorboard --logdir runs --host 0.0.0.0 --port 6006')
for run in ['base', 'finetune']:
    event_files = sorted((Path('runs') / run).glob('events.out.tfevents.*'))
    print(run, 'event files:', len(event_files))
    if event_files:
        accumulator = EventAccumulator(str(event_files[-1]))
        accumulator.Reload()
        print(run, sorted(accumulator.Tags().get('scalars', [])))


tensorboard --logdir runs --host 0.0.0.0 --port 6006
base event files: 5
base ['base/test/accuracy', 'base/test/f1_macro', 'base/test/loss', 'base/test/precision_macro', 'base/test/recall_macro', 'base/train/accuracy', 'base/train/f1_macro', 'base/train/loss', 'base/train/precision_macro', 'base/train/recall_macro']
finetune event files: 5
finetune ['finetune/test/accuracy', 'finetune/test/f1_macro', 'finetune/test/loss', 'finetune/test/precision_macro', 'finetune/test/recall_macro', 'finetune/train/accuracy', 'finetune/train/f1_macro', 'finetune/train/loss', 'finetune/train/precision_macro', 'finetune/train/recall_macro']


## 8. Проверка S3-артефактов

MinIO поднимается через `docker compose up -d`. После обучения модели должны появиться в bucket `mlops-homework`: `models/base_model.pt` и `models/finetuned_model.pt`.

In [9]:
print('MinIO console: http://localhost:9001')
print('bucket:', params['s3']['bucket'])

if params['s3'].get('enabled', False):
    client = create_s3_client(params['s3'])
    response = client.list_objects_v2(Bucket=params['s3']['bucket'], Prefix='models/')
    s3_objects = [
        {'key': item['Key'], 'size_bytes': item['Size']}
        for item in response.get('Contents', [])
    ]
    display(pd.DataFrame(s3_objects))
else:
    print('S3 logging is disabled in params.yaml')


MinIO console: http://localhost:9001
bucket: mlops-homework


,key,size_bytes
0,models/base_model.pt,44791435
1,models/finetuned_model.pt,44791691


## 9. DVC-пайплайн

Файл `dvc.yaml` описывает три стадии: обучение базовой модели, дообучение и сравнение версий.

In [10]:
with open('dvc.yaml', 'r', encoding='utf-8') as file:
    print(file.read())

stages:
  train_base:
    cmd: TORCH_HOME=cache/torch python src/run_training.py --stage base --params params.yaml
    deps:
      - src/mlops_utils.py
      - src/run_training.py
      - params.yaml
      - ai-vs-human-generated-dataset-hw/Train_1
      - ai-vs-human-generated-dataset-hw/Test_1
    params:
      - project
      - model
      - s3
      - train_base
    outs:
      - artifacts/base_model.pt
    metrics:
      - reports/base_metrics.json:
          cache: false

  finetune:
    cmd: TORCH_HOME=cache/torch python src/run_training.py --stage finetune --params params.yaml
    deps:
      - src/mlops_utils.py
      - src/run_training.py
      - params.yaml
      - ai-vs-human-generated-dataset-hw/Train_2
      - ai-vs-human-generated-dataset-hw/Test_2
      - artifacts/base_model.pt
    params:
      - project
      - model
      - s3
      - finetune
    outs:
      - artifacts/finetuned_model.pt
    metrics:
      - reports/finetuned_metrics.json:
          cache: false



Команды для запуска DVC-пайплайна и печати графа:

In [11]:
print('dvc init')
print('dvc repro')
print('dvc dag')

path = Path('reports/dvc_dag.txt')
if path.exists():
    print(path.read_text())


dvc init
dvc repro
dvc dag
          +------------+       
          | train_base |       
          +------------+       
           ***        ***      
          *              *     
        **                ***  
+----------+                 * 
| finetune |              ***  
+----------+             *     
           ***        ***      
              *      *         
               **  **          
            +---------+        
            | compare |        
            +---------+        



## 10. Итоговый вывод

In [12]:
base = read_json('reports/base_metrics.json')
finetuned = read_json('reports/finetuned_metrics.json')
comparison = read_json('reports/comparison.json')

summary = pd.DataFrame([
    {
        'model_version': 'base',
        'train_split': base['train_split'],
        'test_split': base['test_split'],
        'test_loss': base['final_test_metrics']['loss'],
        'test_accuracy': base['final_test_metrics']['accuracy'],
        'test_f1_macro': base['final_test_metrics']['f1_macro'],
        'test_precision_macro': base['final_test_metrics']['precision_macro'],
        'test_recall_macro': base['final_test_metrics']['recall_macro'],
    },
    {
        'model_version': 'finetuned',
        'train_split': finetuned['train_split'],
        'test_split': finetuned['test_split'],
        'test_loss': finetuned['final_test_metrics']['loss'],
        'test_accuracy': finetuned['final_test_metrics']['accuracy'],
        'test_f1_macro': finetuned['final_test_metrics']['f1_macro'],
        'test_precision_macro': finetuned['final_test_metrics']['precision_macro'],
        'test_recall_macro': finetuned['final_test_metrics']['recall_macro'],
    },
])
summary

,model_version,train_split,test_split,test_loss,test_accuracy,test_f1_macro,test_precision_macro,test_recall_macro
0,base,Train_1,Test_1,0.101524,0.963473,0.963466,0.963776,0.963457
1,finetuned,Train_2,Test_2,0.089752,0.971000,0.970987,0.971553,0.970919


In [13]:
print(comparison['conclusion'])

После дообучения качество на тестовой выборке выросло по macro F1. Это означает, что модель смогла использовать данные Train_2 и лучше обобщается на Test_2.


Итог: в работе реализованы обучение модели, логирование метрик в TensorBoard, сохранение двух версий модели в локальный S3 MinIO, DVC-пайплайн для воспроизведения обучения/дообучения/сравнения и таблица сравнения качества двух версий модели.